# InterPro domain labeling for the specificity test set

In [ ]:
import json
import pathlib
import re

import pandas as pd

In [ ]:
def clean_label(value):
    """
    Normalize whitespace/underscores/commas and treat empty strings as None.
    """
    if value is None:
        return None
    value = value.strip()
    if not value:
        return None
    
    value = value.replace("_", " ")
    value = re.sub(r"\s+", " ", value)
    value = re.sub(r"\s*,\s*", ", ", value)

    return value.strip()

def protein_sequences_from_csv(sequence_path):
    """
    Load the CSV at `sequence_path` and return a list of unique, protein
    sequences.
    """
    sequences_df = pd.read_csv(sequence_path)
    protein_rows = sequences_df.loc[
        sequences_df["chain_type"] == "polypeptide(L)"
    ]

    protein_sequences = protein_rows["sequence"]

    protein_sequences = protein_sequences.dropna()
    protein_sequences = protein_sequences.drop_duplicates()

    return protein_sequences.tolist()

def load_evaluation_df():
    evaluation_df = pd.read_csv("../evaluation_csvs/specificity_test.csv")
    return evaluation_df[
        evaluation_df.dataset_name != "rf2na_distillation_transfac"
    ]

def join_unique(values):
    """
    Return sorted, unique, non-empty strings joined by semicolons.
    """
    return ";".join(sorted({value for value in values if value}))

def find_cluster_root(parent_indices, hit_index):
    """
    Follow union-find parent pointers until reaching a cluster root.
    """
    while parent_indices[hit_index] != hit_index:
        hit_index = parent_indices[hit_index]
    return hit_index

def compute_unique_span_coverage(spans):
    """
    Return the number of uniquely covered residue positions across spans.
    """
    if not spans:
        return 0

    sorted_spans = sorted(spans)
    merged_start, merged_end = sorted_spans[0]
    unique_coverage = 0

    for start, end in sorted_spans[1:]:
        if start <= merged_end + 1:
            merged_end = max(merged_end, end)
            continue

        unique_coverage += merged_end - merged_start + 1
        merged_start, merged_end = start, end

    unique_coverage += merged_end - merged_start + 1
    return unique_coverage

def build_id_family_mapping(results_dir, evaluation_df):
    """
    Parse InterPro JSON exports into one id -> family row per evaluation
    example.

    Workflow:
    1. Aggregate DOMAIN hits by protein sequence and accession.
    2. Merge near-duplicate hits that cover the same residue region.
    3. Collapse representative domain labels across each example's chains.
    """
    result_paths = sorted(pathlib.Path(results_dir).glob("*.json"))
    if not result_paths:
        raise FileNotFoundError(
            f"Could not find any InterPro JSON exports under {results_dir}/."
        )

    # --- Step 1: per-sequence domain aggregation ---
    # For each protein sequence, collect every DOMAIN-type InterPro hit,
    # grouped by accession. Different databases can report the same accession;
    # we merge those under one entry per accession, tracking which databases
    # agreed and the residue-level location fragments (used later to cluster
    # overlapping hits).
    # sequence -> {accession -> {label, accession, libraries, spans}}
    sequence_to_domains = {}
    for result_path in result_paths:
        # Load the result JSON.
        with result_path.open("r", encoding="utf-8") as handle:
            payload = json.load(handle)

        # Step through each result.
        for result in payload.get("results", []):
            # Skip results without a sequence or any matches.
            sequence = result.get("sequence")
            if not sequence:
                continue

            # For each match, skip non-DOMAIN signatures and aggregate the rest
            # by sequence and accession.
            domains_by_accession = sequence_to_domains.setdefault(sequence, {})
            for match in result.get("matches", []):
                # Select the associated signature, and extract the InterPro
                # entry if it exists; otherwise, fall back to the signature.
                signature = match["signature"]
                source = signature.get("entry") or signature

                # Skip non-DOMAIN examples.
                if source.get("type") != "DOMAIN":
                    continue

                # Skip examples without an accession.
                accession = source.get("accession")
                if not accession:
                    continue

                # Grab the label and library.
                label = clean_label(
                    source.get("description") or source.get("name")
                ) or accession
                library = signature["signatureLibraryRelease"]["library"]

                # Update the hit for this accession.
                domain_hit = domains_by_accession.setdefault(accession, {
                    "accession": accession,
                    "label": label,
                    "libraries": set(),
                    "spans": [],
                })

                # More descriptive labels preferred.
                if len(label) > len(domain_hit["label"]):
                    domain_hit["label"] = label
  
                # Record the library.
                domain_hit["libraries"].add(library)

                # Record the start/end of each location fragment.
                for location in match.get("locations", []):
                    for fragment in location["location-fragments"]:
                        domain_hit["spans"].append(
                            (fragment["start"], fragment["end"])
                        )

    # --- Step 2: cluster each sequence's hits by >=90% containment ---
    # InterPro often reports several sibling signatures for the same residue
    # region. Their labels and exact boundaries may differ slightly, but the
    # underlying biological domain is the same. We collapse these near-duplicate
    # calls with a union-find merge over accession-level hits.
    sequence_to_representative_hits = {}
    for sequence, domains_by_accession in sequence_to_domains.items():
        # Convert each accession-level record into one span summary so overlap
        # comparisons happen on the full covered region, not on individual
        # fragments.
        domain_hits = [
            {
                **domain_hit,
                "start": min(start for start, _ in domain_hit["spans"]),
                "end": max(end for _, end in domain_hit["spans"]),
                "coverage": compute_unique_span_coverage(
                    domain_hit["spans"]
                ),
            }
            for domain_hit in domains_by_accession.values()
        ]

        # Each hit starts in its own cluster; clusters are merged when two hits
        # overlap across at least 90% of the shorter span.
        cluster_parent_indices = list(range(len(domain_hits)))
        for first_hit_index, first_hit in enumerate(domain_hits):
            for second_hit_index in range(
                first_hit_index + 1, len(domain_hits)
            ):
                second_hit = domain_hits[second_hit_index]

                # Measure the overlap against the shorter hit so nested calls
                # merge, while neighboring domains that only partially overlap
                # stay separate.
                overlap_start = max(first_hit["start"], second_hit["start"])
                overlap_end = min(first_hit["end"], second_hit["end"])
                overlap_length = max(overlap_end - overlap_start + 1, 0)
                shorter_span_length = min(
                    first_hit["end"] - first_hit["start"] + 1,
                    second_hit["end"] - second_hit["start"] + 1,
                )

                if overlap_length / shorter_span_length < 0.9:
                    continue

                first_root = find_cluster_root(
                    cluster_parent_indices,
                    first_hit_index,
                )
                second_root = find_cluster_root(
                    cluster_parent_indices,
                    second_hit_index,
                )
                cluster_parent_indices[first_root] = second_root

        # Group hits by their final cluster root so each group represents one
        # putative residue region.
        hits_by_cluster_root = {}
        for hit_index, domain_hit in enumerate(domain_hits):
            cluster_root = find_cluster_root(cluster_parent_indices, hit_index)
            hits_by_cluster_root.setdefault(cluster_root, []).append(domain_hit)

        # Keep one representative hit per region. Preference order:
        # 1. Supported by the most InterPro libraries.
        # 2. Covers the most residues across fragments.
        # 3. Has the most descriptive human-readable label.
        sequence_to_representative_hits[sequence] = [
            max(
                cluster_hits,
                key=lambda cluster_hit: (
                    len(cluster_hit["libraries"]),
                    cluster_hit["coverage"],
                    len(cluster_hit["label"]),
                ),
            )
            for cluster_hits in hits_by_cluster_root.values()
        ]

    # --- Step 3: one row per evaluation id ---
    # Each evaluation example can include several protein chains. For the final
    # export, we gather the representative domain labels from every chain in the
    # example, deduplicate them, and write a semicolon-delimited summary in the
    # `family` column.
    id_family_rows = []
    for _, evaluation_row in evaluation_df.iterrows():
        # Reload the chain table for this example and keep only unique protein
        # sequences so repeated chains do not duplicate family labels.
        protein_sequences = protein_sequences_from_csv(
            evaluation_row.sequences_path
        )

        # Gather the representative hits for each protein chain in the
        # example into one flat list.
        representative_hits = []
        for protein_sequence in protein_sequences:
            chain_representative_hits = sequence_to_representative_hits.get(
                protein_sequence,
                [],
            )
            representative_hits.extend(chain_representative_hits)

        # Keep only the readable labels in the exported CSV.
        id_family_rows.append({
            "id": evaluation_row.id,
            "family": join_unique(
                representative_hit["label"]
                for representative_hit in representative_hits
            ),
        })
        
    return id_family_rows


## Build FASTAs

In [ ]:
fasta_dir = pathlib.Path("fastas")
fasta_dir.mkdir(exist_ok=True)

evaluation_df = load_evaluation_df()

# Assign one sequential SEQ_XXXX header per unique protein sequence across 
# all examples.
sequence_to_header = {}
for _, row in evaluation_df.iterrows():
    for sequence in protein_sequences_from_csv(row.sequences_path):
        sequence_to_header.setdefault(
            sequence, 
            f"SEQ_{len(sequence_to_header) + 1:04d}"
        )

# Write batches of up to 100 sequences to match the InterPro web UI limit.
sequences_and_headers = list(sequence_to_header.items())
for batch_index in range(0, len(sequences_and_headers), 100):
    batch_path = fasta_dir / f"batch_{batch_index // 100:02d}.fasta"
    with batch_path.open("w") as batch_file:
        for sequence, header in sequences_and_headers[
            batch_index : batch_index + 100
        ]:
            batch_file.write(f">{header}\n{sequence}\n")

print(f"Wrote {len(sequences_and_headers)} unique sequences to ./{fasta_dir}")

## Parse results

In [ ]:
specificity_evaluation_df = load_evaluation_df()

# Parse the InterPro exports and build one id -> family mapping row per
# evaluation example.
id_family_mapping_df = pd.DataFrame(
    build_id_family_mapping("results", specificity_evaluation_df)
)
id_family_mapping_df.to_csv("id_family_mapping.csv", index=False)